# GalaxyZoo Classifier - End-to-End Pipeline
This notebook demonstrates the data loading, training, evaluation, and feature visualization for the GalaxyZoo Classifier.
There are 3 main training options available in this pipeline:
1. Baseline CNN
2. Training SimCLR
3. Finetuning SimCLR weights

## 1. Setup and Imports
We import all the necessary visualization and training utility functions from the `src/` modularized scripts.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.train import load_config, create_dataloader, train_one_epoch, validate
from src.train_ssl import create_ssl_dataloader, train_one_epoch_simclr, validate_simclr
from src.utils import plot_training_curves, plot_confusion_matrix, get_all_predictions, print_performance_report
from src.dataset import GalaxyDataset
from src.architectures import CNNclassifier, SimCLRModel, Image_Augmentations

## 2. Load Configuration and Data
Load parameters from `configs/config.yaml` to set up dataloaders.

In [ ]:
file_path = load_config('Data Parameters', 'file')
batch_size = load_config('Training Parameters', 'batch_size')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

train_loader, val_loader, test_loader, weights = create_dataloader(file_path, batch_size)
print(f"Data loaded successfully with batch size {batch_size}")

## 3. Visualize Generated Augmentations
For Self-Supervised Learning (SimCLR), each image is augmented twice to produce positive pairs. Let's see some examples.

In [ ]:
from src.visualize_augs import main as show_augmentations
# This function will output 'augmentations_preview.png' demonstrating the views
show_augmentations()

## 4. Option 1: Train Baseline CNN
Train a conventional ResNet-based CNN from scratch on the GalaxyZoo images.

In [ ]:
# To train the baseline CNN, uncomment and run:
# from src.train import main as train_baseline
# train_baseline()

## 5. Option 2: Train SimCLR Representations
Pre-train the backbone using Contrastive Learning (SimCLR) without labels.

In [ ]:
# To train the SimCLR model from scratch, uncomment and run:
# from src.train_ssl import main_ssl
# main_ssl()

## 6. Option 3: Finetuning SimCLR Weights
Load the pre-trained SimCLR backbone, append a classification head, and fine-tune.

In [ ]:
# Fine-tuning is typically triggered by setting `loading_weights: True` in config.yaml
# and executing the main training script. Uncomment and run:
# from src.train import main as finetune
# finetune()

## 7. Option 4: Load Existing Model and Show Metrics
Instead of training from scratch, we can load an already trained model to evaluate its metrics on the validation dataset. You can toggle between evaluating the **Baseline CNN** or the **Finetuned SimCLR model**.

In [ ]:
# Choose which model to evaluate: 'baseline' or 'finetuned'
model_type = 'finetuned' 

if model_type == 'baseline':
    model_path = 'models/CNN model/best_model_supervised_CNN.pth'
elif model_type == 'finetuned':
    model_path = 'models/SimCLR training/best_finetuned_model.pth'

# Load best model weights to evaluate:
model = CNNclassifier()
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

class_names = [f'Class {i}' for i in range(10)]
y_true, y_pred = get_all_predictions(model, val_loader, device)

print_performance_report(y_true, y_pred, class_names)
plot_confusion_matrix(y_true, y_pred, class_names, save_path=f'{model_type}_latest_confusion_matrix.png')

## 8. Feature Visualizations
Visualize what the network learned using Grad-CAM, Feature Maps, and Nearest Neighbors.

In [ ]:
from src.visualize_features import load_backbone, get_test_loader, visualize_gradcam, visualize_feature_maps, visualize_nearest_neighbors

# 1. Load backbone and data
backbone = load_backbone(device)
test_loader = get_test_loader(batch_size=64)

# 2. Visualize Grad-CAM
visualize_gradcam(backbone, test_loader, device)

# 3. Visualize Feature Maps
visualize_feature_maps(backbone, test_loader, device)

# 4. Visualize Nearest Neighbors
visualize_nearest_neighbors(backbone, test_loader, device)